# EXP-128 · Gemma 4 31B QLoRA(EXP-125) 선지 순서 이동 TTA (A100, 학습 없음)

EXP-125에서 학습한 LoRA(마지막 체크포인트)를 불러와 **선지를 2칸 순환 이동(a,b,c,d → c,d,a,b)** 한 상태로 한 번 더 풀고, 원래 순서로 되돌려 저장합니다.
앙상블에서 EXP-125(원래 순서)와 평균하면 위치 편향이 상쇄됩니다 (EXP-123 Qwen3.5와 같은 방식).

순서: 설치 → 데이터 → 모델(체크포인트 자동 로드, `재개: .../step_320` 확인) → 프롬프트 → 점검 → valid → test

In [ ]:
# ===== 0. 설치 (실행 후 '런타임 → 세션 다시 시작') =====
!pip install -q -U transformers accelerate peft bitsandbytes
!pip uninstall -y -q torchao
!apt-get install -y -qq p7zip-full > /dev/null

In [ ]:
# ===== 1. 드라이브 · 데이터 =====
import os, glob, shutil, subprocess, time, random, math
import numpy as np, pandas as pd
from google.colab import drive
drive.mount("/content/drive")
# 공유 폴더 위치 자동 탐색 (내 드라이브/ssafy/ai-challenge 또는 바로가기 내 드라이브/ai-challenge)
cands = ["/content/drive/MyDrive/ssafy/ai-challenge", "/content/drive/MyDrive/ai-challenge"] + glob.glob("/content/drive/MyDrive/*/ai-challenge")
SHARE_DIR = next((c for c in cands if os.path.exists(f"{c}/split/valid_split_seed43.csv")), None)
assert SHARE_DIR, "ai-challenge 폴더를 찾지 못했습니다. 공유 폴더를 '내 드라이브'에 바로가기로 추가했는지 확인하세요."
print("SHARE_DIR =", SHARE_DIR)
EXP_DIR = f"{SHARE_DIR}/experiment_data"
DATA_ROOT = "/content/data"
cnt = lambda d: len(glob.glob(f"{DATA_ROOT}/{d}/*.jpg"))
if cnt("train") < 6714 or cnt("test") < 6714:
    shutil.copy(glob.glob(f"{SHARE_DIR}/ssafy-16-2-ai*.zip")[0], "/content/data.zip")
    shutil.rmtree(DATA_ROOT, ignore_errors=True); os.makedirs(DATA_ROOT)
    subprocess.run(["7z","x","-y","-pocrisnotenough",f"-o{DATA_ROOT}","/content/data.zip"], check=True, stdout=subprocess.DEVNULL)
print("이미지 수 → train:", cnt("train"), "test:", cnt("test"))
resolve = lambda p: p if os.path.isabs(str(p)) else os.path.join(DATA_ROOT, str(p))
COLS = ["id","path","question","a","b","c","d","answer"]

valid_df = pd.read_csv(f"{SHARE_DIR}/split/valid_split_seed43.csv")[COLS]
clean = pd.concat([pd.read_csv(f"{SHARE_DIR}/split/train_clean_v2_seed43.csv"), pd.read_csv(f"{SHARE_DIR}/split/valid_clean_v2_seed43.csv")])[COLS]
train_df = clean[~clean.id.isin(set(valid_df.id))].reset_index(drop=True)
test_df = pd.read_csv(f"{DATA_ROOT}/test.csv")
for d in (valid_df, train_df, test_df): d["path"] = d["path"].map(resolve)
assert not set(train_df.id) & set(valid_df.id)

SEED = 43; rng = np.random.default_rng(SEED)
groups = {a: list(rng.permutation(train_df.index[train_df.answer == a])) for a in "abcd"}
order_all = []
while any(groups.values()):
    for a in "abcd":
        if groups[a]: order_all.append(groups[a].pop())
print("train 후보", len(train_df), "| valid", len(valid_df), "| test", len(test_df))

In [ ]:
# ===== 2. 모델 (다운로드 약 62GB, 5~10분) =====
import torch, torch.nn.functional as F
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig, get_cosine_schedule_with_warmup
from peft import LoraConfig, get_peft_model, PeftModel
Image.MAX_IMAGE_PIXELS = None

MODEL_ID = "google/gemma-4-31B-it"
TAG = "EXP-125_gemma4-31b_qlora"   # 체크포인트 위치 (학습 결과 재사용)
OUT_TAG = "EXP-128_gemma4-31b_qlora_shift2"
CKPT_DIR = f"{SHARE_DIR}/saved_models/{TAG}"
SOFT_TOKENS = 560
random.seed(SEED); torch.manual_seed(SEED)

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16)
model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, quantization_config=bnb, device_map="cuda", dtype=torch.bfloat16, attn_implementation="sdpa")
processor = AutoProcessor.from_pretrained(MODEL_ID, padding_side="left", max_soft_tokens=SOFT_TOKENS)
tok = processor.tokenizer
# 메모리 절약: prepare_model_for_kbit_training(임베딩 fp32 변환, 약 +6GB) 대신 체크포인팅만 켬
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.enable_input_require_grads()
model.config.use_cache = False

TARGETS = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]
START_STEP = 0
latest = sorted([p for p in glob.glob(f"{CKPT_DIR}/step_*") if os.path.exists(f"{p}/adapter_model.safetensors")], key=lambda p: int(p.split("_")[-1]))
if latest:
    model = PeftModel.from_pretrained(model, latest[-1], is_trainable=True); START_STEP = int(latest[-1].split("_")[-1]); print("재개:", latest[-1])
else:
    lora_kw = dict(r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM")
    try:   # 언어 모델 층만 (비전 인코더 제외)
        model = get_peft_model(model, LoraConfig(target_modules=r".*language_model.*\.(" + "|".join(TARGETS) + ")", **lora_kw))
    except ValueError as e:
        print("language_model 경로 없음 → 비전 제외 정규식으로 재시도:", str(e)[:120])
        model = get_peft_model(model, LoraConfig(target_modules=r"^(?!.*(vision|audio)).*\.(" + "|".join(TARGETS) + ")", **lora_kw))
model.print_trainable_parameters()
n_lora = sum(1 for n, m in model.named_modules() if hasattr(m, "lora_A"))
print("LoRA 모듈 수:", n_lora, "| 예:", next(n for n, m in model.named_modules() if hasattr(m, "lora_A")))
print("GPU 사용", round(torch.cuda.memory_allocated()/1e9, 1), "GB"); assert n_lora > 0

In [ ]:
# ===== 3. 프롬프트 · 선지 섞기 · 정답 기호 토큰 loss =====
def prompt(q, opts):   # EXP-120과 같은 문구
    return (f"이미지를 자세히 보고 질문에 답하세요. 이미지 속 글자를 정확히 읽는 것이 중요합니다.\n\n"
            f"질문: {q}\n" + "".join(f"{l}. {o}\n" for l, o in zip("abcd", opts)) +
            "\n정답 선지의 알파벳(a, b, c, d) 한 글자만 출력하세요.")

def chat(img, text):
    msgs = [{"role":"user","content":[{"type":"image","image":img},{"type":"text","text":text}]}]
    try: return processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except TypeError: return processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def build_train_example(r, shuffle=True):
    opts = [str(r[c]) for c in "abcd"]; gold = "abcd".index(str(r["answer"]).strip().lower())
    order = list(range(4))
    if shuffle: random.shuffle(order)
    new_opts = [opts[i] for i in order]; new_gold = "abcd"[order.index(gold)]
    img = Image.open(r["path"]).convert("RGB")
    enc = processor(text=[chat(img, prompt(r["question"], new_opts)) + new_gold], images=[img], return_tensors="pt")
    labels = torch.full_like(enc["input_ids"], -100); labels[0, -1] = enc["input_ids"][0, -1]   # 마지막 토큰(정답 기호)에만 loss
    enc["labels"] = labels
    return enc

CHOICE_IDS = []
for ch in "abcd":
    ss = set()
    for x in [ch, ch.upper(), " "+ch, " "+ch.upper()]:
        t = tok.encode(x, add_special_tokens=False)
        if len(t) == 1: ss.add(t[0])
    CHOICE_IDS.append(sorted(ss))
print("선지 토큰:", CHOICE_IDS)

@torch.inference_mode()
def score(r):
    img = Image.open(r["path"]).convert("RGB")
    inp = processor(text=[chat(img, prompt(r["question"], [str(r[c]) for c in "abcd"]))], images=[img], return_tensors="pt").to("cuda")
    with torch.autocast("cuda", dtype=torch.bfloat16):
        try: lg = model(**inp, logits_to_keep=1, use_cache=False).logits[0, -1].float()
        except TypeError: lg = model(**inp, use_cache=False).logits[0, -1].float()
    return F.softmax(torch.stack([torch.logsumexp(lg[i], 0) for i in CHOICE_IDS]), 0).cpu().numpy()

In [ ]:
# ===== 4. 선지 이동 채점 + 점검 =====
SHIFT = 2
@torch.inference_mode()
def score_shift(r):
    img = Image.open(r["path"]).convert("RGB")
    orig = [str(r[c]) for c in "abcd"]; order = [(j + SHIFT) % 4 for j in range(4)]   # 표시 위치 j에 원래 선지 order[j]
    inp = processor(text=[chat(img, prompt(r["question"], [orig[i] for i in order]))], images=[img], return_tensors="pt").to("cuda")
    with torch.autocast("cuda", dtype=torch.bfloat16):
        try: lg = model(**inp, logits_to_keep=1, use_cache=False).logits[0, -1].float()
        except TypeError: lg = model(**inp, use_cache=False).logits[0, -1].float()
    p = F.softmax(torch.stack([torch.logsumexp(lg[i], 0) for i in CHOICE_IDS]), 0).cpu().numpy()
    out = np.zeros(4)
    for j, i in enumerate(order): out[i] = p[j]                                        # 원래 선지 순서로 되돌림
    return out
model.eval()
for _, r in valid_df.head(5).iterrows():
    print("정답", r["answer"], "| 원래 순서", score(r).round(2), "| 이동 후 되돌림", score_shift(r).round(2))
print("→ 두 확률의 최댓값 위치가 대부분 같고 정답과 맞으면 정상")

In [ ]:
# ===== 5. valid 672 =====
P = ["prob_a","prob_b","prob_c","prob_d"]
def run(df, out_csv, every=50):
    done = pd.read_csv(out_csv) if os.path.exists(out_csv) else pd.DataFrame(columns=["id"]+P)
    todo = df[~df.id.astype(str).isin(set(done.id.astype(str)))]; rows = []; t0 = time.time()
    print(f"완료 {len(done)} / 남음 {len(todo)}")
    for n, (_, r) in enumerate(todo.iterrows(), 1):
        try: p = score_shift(r)
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache(); p = np.full(4, .25); print("OOM", r.id)
        rows.append([r.id, *p])
        if n % every == 0 or n == len(todo):
            done = pd.concat([done, pd.DataFrame(rows, columns=done.columns)], ignore_index=True); done.to_csv(out_csv, index=False); rows = []
            el = time.time()-t0; print(f"{n}/{len(todo)} {el/n:.2f}s/문항, 남은 {el/n*(len(todo)-n)/60:.0f}분")
    return done.set_index("id")
vp = run(valid_df, f"{EXP_DIR}/{OUT_TAG}_valid_probs.csv")
ids = valid_df.id.values; ans = valid_df.answer.map(dict(zip("abcd", range(4)))).values
s2 = vp.loc[ids, P].values.astype(float)
s0 = pd.read_csv(f"{EXP_DIR}/EXP-125_gemma4-31b_qlora_valid_probs.csv").set_index("id").loc[ids, P].values.astype(float)
ok = lambda p: (p.argmax(1) == ans).sum()
print(f"원래 순서 {ok(s0)} | 2칸 이동 {ok(s2)} | 평균 {ok((s0+s2)/2)} /672 | 두 방식 답이 다른 문항 {(s0.argmax(1)!=s2.argmax(1)).sum()}")

## 6. test 6,714 (끊기면 이 셀만 다시 실행 → 이어서 진행)

In [ ]:
tp = run(test_df, f"{EXP_DIR}/{OUT_TAG}_test_probs.csv", every=100)
print(tp.shape)